# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fatima8211/ML_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
### My rule

I will rank pages higher when they show a combination of meaningful search visibility, possible freshness risk, and a CTR opportunity.

A page receives more score when it has enough previous-30-day impressions to matter, has not been updated recently, and has a relatively low CTR while still appearing in a useful search-position range. The rule is intentionally simple and transparent so a reviewer can understand why a page was ranked.

### Reason codes

The rule uses one reason code per page:

* `stale_visible_page` — the page has meaningful search visibility and has not been updated for at least 180 days.
* `low_ctr_visible_page` — the page has meaningful visibility, ranks within positions 1–20, and has CTR below 0.5.
* `position_opportunity` — the page has meaningful visibility and is ranking outside the strongest first-page range.
* `monitor` — the page does not meet one of the stronger review conditions.

The score is based only on observable information available at the decision point. I do not use `trend_direction`, `trend_pct`, the declining proxy, client identity, or any future-window outcome as an input.


In [1]:
# This cell is for CODE (numbers, a query, a check).
import os
import numpy as np
import pandas as pd

# Load the starter dataset
csv_path = "data/raw/content_refresh_anonymized.csv"

if not os.path.exists(csv_path):
    # Useful when running directly in Colab from the GitHub repository
    csv_path = (
        "https://raw.githubusercontent.com/Fatima8211/ML_Internship/"
        "main/data/raw/content_refresh_anonymized.csv"
    )

df = pd.read_csv(csv_path)

# Define the proxy only for evaluation.
# It is NOT used as an input feature for the score.
df["declining_proxy"] = (
    df["impressions_last_30d"] < 0.8 * df["impressions_prev_30d"]
).astype(int)

# Basic checks
required_cols = [
    "content_id",
    "impressions_prev_30d",
    "days_since_last_update",
    "ctr",
    "avg_position",
]

missing = [c for c in required_cols if c not in df.columns]
assert not missing, f"Missing required columns: {missing}"

print("Rows:", len(df))
print("Unique content pages:", df["content_id"].nunique())
print("Required columns available:", True)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Rows: 30000
Unique content pages: 30000
Required columns available: True


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Ranked queue

The score combines three simple observable conditions.

First, a page needs enough previous-period impressions to have meaningful visibility. Second, stale pages receive additional priority because an old page with meaningful exposure is a practical refresh candidate. Third, pages with low CTR in positions 1–20 receive additional priority because they have an observable click-through opportunity.

The score is deliberately hand-written rather than fitted. Higher scores mean "review earlier," not "this page definitely needs a refresh."

The queue will contain one reason code and one action label for every page.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Create baseline signals
df["visible"] = df["impressions_prev_30d"] >= 500

df["stale"] = df["days_since_last_update"] >= 180

df["low_ctr"] = (
    (df["ctr"] > 0) &
    (df["ctr"] < 0.5) &
    (df["avg_position"] > 0) &
    (df["avg_position"] <= 20)
)

df["position_opportunity"] = (
    df["visible"] &
    (df["avg_position"] > 10) &
    (df["avg_position"] > 0)
)

# Transparent baseline score
df["score"] = (
    df["visible"].astype(int)
    + 2 * (df["visible"] & df["stale"]).astype(int)
    + 2 * (df["visible"] & df["low_ctr"]).astype(int)
    + df["position_opportunity"].astype(int)
)

# Reason code
def get_reason(row):
    if row["visible"] and row["stale"]:
        return "stale_visible_page"
    elif row["visible"] and row["low_ctr"]:
        return "low_ctr_visible_page"
    elif row["position_opportunity"]:
        return "position_opportunity"
    else:
        return "monitor"

df["reason_code"] = df.apply(get_reason, axis=1)

# Action
df["action"] = np.where(
    df["score"] >= 3,
    "review_refresh",
    np.where(
        df["score"] >= 1,
        "review",
        "monitor"
    )
)

# Rank highest priority first
ranked_df = (
    df.sort_values(
        ["score", "impressions_prev_30d"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

ranked_df["rank"] = ranked_df.index + 1

# Display top 20
display(
    ranked_df[
        [
            "rank",
            "content_id",
            "score",
            "action",
            "reason_code",
            "impressions_prev_30d",
            "days_since_last_update",
            "ctr",
            "avg_position"
        ]
    ].head(20)
)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


,rank,content_id,score,action,reason_code,impressions_prev_30d,days_since_last_update,ctr,avg_position
0,1,content_cf56e2e2e282,6,review_refresh,stale_visible_page,26791,194,0.15,19.7
1,2,content_0a91db491d14,6,review_refresh,stale_visible_page,4662,193,0.49,10.5
2,3,content_c2d929d83eaa,6,review_refresh,stale_visible_page,2160,193,0.20,17.9
3,4,content_fe16a55cd13d,6,review_refresh,stale_visible_page,1562,194,0.33,16.4
4,5,content_928af3e22c80,6,review_refresh,stale_visible_page,702,193,0.12,15.8
5,6,content_77d4d5930e5e,6,review_refresh,stale_visible_page,527,194,0.24,18.6
6,7,content_c5063073d048,4,review_refresh,low_ctr_visible_page,65696,104,0.24,12.5
7,8,content_eb366e871254,4,review_refresh,low_ctr_visible_page,58717,104,0.20,16.6
8,9,content_758db544d84f,4,review_refresh,low_ctr_visible_page,45777,104,0.41,13.8
9,10,content_b9f7afeded79,4,review_refresh,low_ctr_visible_page,38962,104,0.31,19.1


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

I review the highest-ranked pages manually rather than treating the score as a final decision.

The confidence note describes how strong the observable evidence is. A page with meaningful visibility and multiple supporting signals receives a stronger confidence note than a page supported by only one signal.

For each page, I also state what could make the recommendation wrong. Possible reasons include low traffic volume, noisy CTR, temporary search variation, seasonality, a position that does not represent the page's main queries, or a stale-page signal that does not imply a refresh would help.

The review is decision-support, not an automatic refresh instruction.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Section 3 — Top-20 Review

top20 = ranked_df.head(20).copy()

def confidence_note(row):
    if row["score"] >= 4:
        return "Higher confidence: multiple observable risk/opportunity signals agree."
    elif row["score"] >= 2:
        return "Moderate confidence: at least one meaningful signal supports review."
    else:
        return "Lower confidence: limited signal evidence; manual review is important."

def what_would_make_it_wrong(row):
    if row["reason_code"] == "stale_visible_page":
        return "The page may be intentionally evergreen, recently improved, or not in need of a refresh."
    elif row["reason_code"] == "low_ctr_visible_page":
        return "Low CTR may be explained by search intent, SERP features, or a query mix that is not actionable."
    elif row["reason_code"] == "position_opportunity":
        return "The page may have suitable rankings but insufficient demand or a non-actionable query mix."
    else:
        return "The baseline may miss an important signal or the observed weakness may be temporary."

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(
    what_would_make_it_wrong,
    axis=1
)

review_table = top20[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(review_table)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


,rank,content_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_cf56e2e2e282,review_refresh,stale_visible_page,Higher confidence: multiple observable risk/op...,"The page may be intentionally evergreen, recen..."
1,2,content_0a91db491d14,review_refresh,stale_visible_page,Higher confidence: multiple observable risk/op...,"The page may be intentionally evergreen, recen..."
2,3,content_c2d929d83eaa,review_refresh,stale_visible_page,Higher confidence: multiple observable risk/op...,"The page may be intentionally evergreen, recen..."
3,4,content_fe16a55cd13d,review_refresh,stale_visible_page,Higher confidence: multiple observable risk/op...,"The page may be intentionally evergreen, recen..."
4,5,content_928af3e22c80,review_refresh,stale_visible_page,Higher confidence: multiple observable risk/op...,"The page may be intentionally evergreen, recen..."
5,6,content_77d4d5930e5e,review_refresh,stale_visible_page,Higher confidence: multiple observable risk/op...,"The page may be intentionally evergreen, recen..."
6,7,content_c5063073d048,review_refresh,low_ctr_visible_page,Higher confidence: multiple observable risk/op...,"Low CTR may be explained by search intent, SER..."
7,8,content_eb366e871254,review_refresh,low_ctr_visible_page,Higher confidence: multiple observable risk/op...,"Low CTR may be explained by search intent, SER..."
8,9,content_758db544d84f,review_refresh,low_ctr_visible_page,Higher confidence: multiple observable risk/op...,"Low CTR may be explained by search intent, SER..."
9,10,content_b9f7afeded79,review_refresh,low_ctr_visible_page,Higher confidence: multiple observable risk/op...,"Low CTR may be explained by search intent, SER..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak-pick candidates

The weak-pick candidates are pages that receive a low baseline score but are still present in the ranked queue. These pages should be reviewed carefully because a simple rule may prioritize pages that look weak based on observable signals but do not actually need an SEO or content intervention.

Possible reasons a pick could be wrong include temporary traffic changes, search-intent differences, SERP features, seasonal demand, or a page that is intentionally evergreen.

### Leakage check

The baseline score uses only observable historical/current-page signals: `impressions_prev_30d`, `days_since_last_update`, `ctr`, and `avg_position`.

The declining proxy, `trend_direction`, `trend_pct`, `impressions_last_30d`, and `client_id` are not used as scoring features. The declining proxy is used only for evaluation, not for ranking the pages.

Therefore, the baseline does not use the future/target information to create the ranking.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# -----------------------------
# Weak picks + leakage check
# -----------------------------

# Weak picks = top-ranked rows with relatively little supporting evidence.
# Section 4 — Weak Picks + Leakage Check

# Make sure rank exists
if "rank" not in ranked_df.columns:
    ranked_df = ranked_df.reset_index(drop=True)
    ranked_df["rank"] = ranked_df.index + 1

# Weak-pick candidates:
# Low-score pages that still entered the ranked review queue.
weak_picks = ranked_df[
    ranked_df["score"] <= 1
].head(10).copy()

print("Weak-pick candidates:")

display(
    weak_picks[
        [
            "rank",
            "content_id",
            "score",
            "action",
            "reason_code",
            "impressions_prev_30d",
            "days_since_last_update",
            "ctr",
            "avg_position"
        ]
    ]
)

# Explicit leakage check.
# These columns must NOT influence the score.
# Leakage check

leakage_columns = {
    "declining_proxy",
    "trend_direction",
    "trend_pct",
    "client_id",
    "impressions_last_30d"
}

score_columns = {
    "impressions_prev_30d",
    "days_since_last_update",
    "ctr",
    "avg_position"
}

print("Leakage check:")
print("Potential leakage columns:", sorted(leakage_columns))

used_for_score = {
    "impressions_prev_30d",
    "days_since_last_update",
    "ctr",
    "avg_position"
}

leaked = used_for_score.intersection(leakage_columns)

if leaked:
    print("WARNING — leakage detected:", leaked)
else:
    print("PASS — no target/future-window/client columns were used in the baseline score.")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Weak-pick candidates:


,rank,content_id,score,action,reason_code,impressions_prev_30d,days_since_last_update,ctr,avg_position
9726,9727,content_9532f197bbc8,1,review,monitor,174235,104,0.87,2.0
9727,9728,content_2c2606c5d176,1,review,monitor,164079,104,0.53,4.2
9728,9729,content_44e481c8f55b,1,review,monitor,118137,20,0.65,1.4
9729,9730,content_c8e9d6ab9013,1,review,monitor,111885,104,0.00,9.7
9730,9731,content_89e84d699e9e,1,review,monitor,97854,20,0.89,4.8
9731,9732,content_aa4baf490b43,1,review,monitor,95175,20,0.50,5.9
9732,9733,content_8e7ba84a972b,1,review,monitor,91776,20,0.92,4.8
9733,9734,content_07e0b9af8b1a,1,review,monitor,86438,8,1.94,3.3
9734,9735,content_89fcb6f35525,1,review,monitor,79320,104,0.56,4.7
9735,9736,content_abf535d9051e,1,review,monitor,66303,20,0.92,3.9


Leakage check:
Potential leakage columns: ['client_id', 'declining_proxy', 'impressions_last_30d', 'trend_direction', 'trend_pct']
PASS — no target/future-window/client columns were used in the baseline score.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.